In [ ]:
!pip install --upgrade yt-dlp
!pip install --quiet google-api-python-client

In [2]:
import os
import json
from google.colab import userdata
from googleapiclient.discovery import build
from google.colab import drive
import json
import os
import requests
from IPython.display import display, Image

# Подключение гугл диска

In [ ]:
drive.mount('/content/drive')

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/YouTube_Downloads"
JSON_PATH = f"{DRIVE_DIR}/channels.json"

DEFAULT_CHANNELS = {
    "ted": {
        "handle": "@ted",
        "channel_id": "UCAuUUnT6oDeKwE6v1NGQxug",
        "uploads_id": "UUAuUUnT6oDeKwE6v1NGQxug"
    }
}

os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.exists(JSON_PATH):
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        channels = json.load(f)
    print(f"База каналов загружена из JSON на Диске. Всего каналов: {len(channels)}")
else:
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(DEFAULT_CHANNELS, f, ensure_ascii=False, indent=4)
    channels = DEFAULT_CHANNELS
    print(f"Создан новый файл базы каналов: {JSON_PATH}")

In [ ]:
if 'channels' in locals() and channels:
    print("СПИСОК ДОБАВЛЕННЫХ YOUTUBE каналов")

    channel_keys = list(channels.keys())

    for idx, key in enumerate(channel_keys, 1):
        handle = channels[key]["handle"]
        print(f"[ {idx} ] {key} ({handle})")

    print("-" * 35)
else:
    print("Ошибка: База каналов не найдена")


# Добавление канала

In [6]:
API_KEY = userdata.get('YOUTUBE_API_KEY')
youtube = build('youtube', 'v3', developerKey=API_KEY)

In [ ]:
print("ДОБАВЛЕНИЕ НОВОГО КАНАЛА")
print("Введите хэндл или название канала (например: droider, wylsacom):")
USER_INPUT_HANDLE = input("Поиск канала: ").strip()

if not USER_INPUT_HANDLE:
    print("Ошибка: Вы ничего не ввели!")
else:
    clean_handle = USER_INPUT_HANDLE.lstrip('@')

    print(f"\nПоиск похожих каналов по запросу '{clean_handle}'...")
    search_response = youtube.search().list(
        q=clean_handle,
        type='channel',
        part='snippet',
        maxResults=5
    ).execute()

    search_results = search_response.get('items', [])

    if search_results:
        print("\nНАЙДЕННЫЕ ВАРИАНТЫ")
        for idx, s_item in enumerate(search_results, 1):
            s_title = s_item['snippet']['title']
            s_id = s_item['id']['channelId']
            s_desc = s_item['snippet']['description'][:70]
            avatar_url = s_item['snippet'].get('thumbnails', {}).get('default', {}).get('url')

            print(f"\n[ {idx} ] {s_title}")
            print(f"      ID канала: {s_id}")
            if s_desc:
                print(f"      Описание: {s_desc}...")

            if avatar_url:
                try:
                    img_data = requests.get(avatar_url).content
                    display(Image(data=img_data, width=60))
                except:
                    pass
        print("-" * 50)

        user_choice = input("Введите номер правильного канала (или нажмите Enter для отмены): ")

        if user_choice.strip():
            choice_idx = int(user_choice.strip()) - 1
            if 0 <= choice_idx < len(search_results):
                chosen_item = search_results[choice_idx]
                channel_id = chosen_item['id']['channelId']
                raw_title = chosen_item['snippet']['title']

                id_duplicate_key = None
                if 'channels' in locals() and channels:
                    for key, data in channels.items():
                        if data.get('channel_id') == channel_id:
                            id_duplicate_key = key
                            break

                if id_duplicate_key:
                    print(f"\nЭтот канал УЖЕ есть в вашей базе под именем '{id_duplicate_key}'!")
                    print("Добавление отменено во избежание дубликатов.")
                else:
                    print(f"Подгружаю системные ID для '{raw_title}'...")
                    c_response = youtube.channels().list(
                        part='contentDetails,snippet',
                        id=channel_id
                    ).execute()

                    c_details = c_response['items'][0]

                    uploads_playlist_id = c_details['contentDetails']['relatedPlaylists']['uploads']
                    actual_handle = c_details['snippet'].get('customUrl', f"@{channel_id}")

                    short_name = "".join(x for x in raw_title if x.isalnum()).lower()
                    if not short_name or (short_name in channels):
                        short_name = actual_handle.lstrip('@').lower()

                    channels[short_name] = {
                        "handle": actual_handle,
                        "channel_id": channel_id,
                        "uploads_id": uploads_playlist_id
                    }

                    if 'JSON_PATH' in locals() and os.path.exists(os.path.dirname(JSON_PATH)):
                        with open(JSON_PATH, "w", encoding="utf-8") as f:
                            json.dump(channels, f, ensure_ascii=False, indent=4)
                        print(f"\nНовый канал успешно сохранен на Google Диск!")
                    else:
                        print("\nПредупреждение: Переменная JSON_PATH не найдена, сохранено временно.")

                    print(f"Успешно добавлено!")
                    print(f"Имя в базе: {short_name} ({actual_handle})")
                    print(f"ID плейлиста загрузок: {uploads_playlist_id}")
            else:
                print("Ошибка: Введен неверный номер.")
        else:
            print("Добавление отменено.")
    else:
        print(f"Ошибка: Ни одного похожего канала по запросу '{USER_INPUT_HANDLE}' не найдено.")


# Загрузка видео с youtube

In [ ]:
NUM_VIDEOS_TO_SHOW = 10

try:
    if os.path.exists(JSON_PATH):
        with open(JSON_PATH, "r", encoding="utf-8") as f:
            channels = json.load(f)
    else:
        channels = {}

    if channels:
        print("ВЫБОР КАНАЛА")
        channel_keys = list(channels.keys())

        for idx, key in enumerate(channel_keys, 1):
            print(f"[ {idx} ] {key} ({channels[key]['handle']})")
        print("-" * 40)

        user_choice = input("Введите номер канала, с которым хотите работать: ")
        choice_idx = int(user_choice.strip()) - 1

        if 0 <= choice_idx < len(channel_keys):
            CHOSEN_KEY = channel_keys[choice_idx]
            uploads_playlist_id = channels[CHOSEN_KEY]["uploads_id"]

            print(f"\nАктивный канал: {CHOSEN_KEY} ({channels[CHOSEN_KEY]['handle']})")
            print("=" * 40 + "\n")
        else:
            print("Ошибка: Введен неверный номер из списка.")
            uploads_playlist_id = None
    else:
        print("Ошибка: База блогеров пуста или файл JSON не найден на Диске. Запустите Блок 1 и 2!")
        uploads_playlist_id = None


    if uploads_playlist_id:
        print(f"Получение последних {NUM_VIDEOS_TO_SHOW} видео для выбора...\n")

        playlist_response = youtube.playlistItems().list(
            part='snippet',
            playlistId=uploads_playlist_id,
            maxResults=NUM_VIDEOS_TO_SHOW
        ).execute()

        video_list = []

        print("СПИСК ДОСТУПНЫХ ВИДЕО")
        for idx, item in enumerate(playlist_response['items'], 1):
            snippet = item['snippet']
            title = snippet['title']
            video_id = snippet['resourceId']['videoId']

            video_url = f"https://youtube.com/watch?v={video_id}"
            video_list.append({'title': title, 'url': video_url})

            thumbnails = snippet.get('thumbnails', {})
            img_url = None
            for quality in ['maxres', 'standard', 'high', 'medium', 'default']:
                if quality in thumbnails:
                    img_url = thumbnails[quality]['url']
                    break

            print(f"\nБлок № {idx}")
            print(f"Название: {title}")
            print(f"Ссылка:   {video_url}")

            if img_url:
                try:
                    img_data = requests.get(img_url).content
                    display(Image(data=img_data, width=250))
                except:
                    pass

            print("-" * 50)


        print("\n Введите через запятую номера видео, которые нужно скачать (например: 1, 3):")
        user_input = input("Номера видео: ")

        chosen_indexes = []
        for x in user_input.split(','):
            try:
                num = int(x.strip())
                if 1 <= num <= len(video_list):
                    chosen_indexes.append(num)
            except ValueError:
                continue


        if chosen_indexes:
            print(f"\n Начинаю скачивание видео ({len(chosen_indexes)} шт.)...")
            for idx in chosen_indexes:
                video_to_download = video_list[idx - 1]
                url = video_to_download['url']
                title = video_to_download['title']

                print(f"\n Скачиваю видео №{idx}: {title}...")

                os.system(f'yt-dlp -P "/content/drive/MyDrive/YouTube_Downloads" -f "b" "{url}"')

            print("\n Всё выбранное скачано! Файлы лежат на вашем Google Диске в папке 'YouTube_Downloads'.")
        else:
            print("\n Вы ничего не выбрали или ввели неверные номера. Скачивание отменено.")

except Exception as e:
    print(f"Произошла ошибка во время работы скрипта: {e}")
